# Joy Tactic-Repair Experiment

Drives the same harness as the ElGamal notebook against the **Joy** corpus with synthetic tactic mutations (`joy-tactic-repair`).

Breakage here is *injected*, not real version drift, so it is a weaker signal about repair ability than ElGamal but a much faster regression check: 33 cases, most 2-3 tactics.

The replay-prefix protection (`ProofFile.protected_prefix`, `net_tactics_vs_bootstrap`) does **not** apply to this spec — it is specific to `replay_bootstrap`, and this spec uses mutations. Everything else (seq bounds, smt advice, transport retries, no-op detection) does.

## Setup

Run this first: it puts the repo root on `sys.path`, changes cwd, and loads `.env`.

In [ ]:
import os, sys
from pathlib import Path

# Repo root on the path, cwd there (corpus paths are root-relative).
_root = Path(os.getcwd())
if _root.name == "notebooks":
    _root = _root.parent
sys.path.insert(0, str(_root))
os.chdir(_root)

import logging
from integration.experiment import notebook_support as ns

keys = ns.load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
# The embeddings endpoint is called thousands of times per trial; its request
# log drowns everything else.
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

print(f"Project root      : {ns.project_root()}")
for k, v in keys.items():
    print(f"{k:<18}: {'set' if v else 'NOT SET'}")


## Verify the kernel is running the code on disk

In [ ]:
# Fails loudly on a stale kernel. New modules are invisible to a running
# kernel regardless of %autoreload, and runs C-F of the ElGamal experiment all
# executed a 22-hour-old loop.py while the operator believed otherwise.
stale = ns.verify_working_tree_is_live()
if stale:
    raise RuntimeError("STALE KERNEL - restart it. Failed: " + ", ".join(stale))
print("OK - working tree code is live")


## Preflight

In [ ]:
from integration.agent.config import AgentConfig

_p = ns.preflight(AgentConfig())
print(f"Embeddings endpoint : {'OK' if _p['embeddings_ok'] else 'UNAVAILABLE'} — {_p['embeddings_detail']}")
if not _p["embeddings_ok"]:
    print("  -> start LM Studio and load an embedding model; EVERY provider needs it.")
print(f"EasyCrypt binary    : {'found' if _p['easycrypt_found'] else 'MISSING'} ({_p['easycrypt_bin']})")
print(f"EasyCrypt version   : {_p['ec_version']} (via {_p['ec_version_method']}, {_p['ec_version_confidence']})")


## Configuration

In [ ]:
# --- Spec -------------------------------------------------------------------
# replay-until-failure. NOTE: every Joy proof compiles AND closes against
# the current build, so replay reaches the end and the agent is never
# invoked. That makes this a REGRESSION CHECK (does the replay path still
# complete cleanly?), not a test of repair ability -- and it does NOT
# exercise the prefix clamp. Use `joy-tactic-repair` for injected
# breakage, or lq1/elgamal changelog-repair for the clamp.
SPEC_NAME = "joy-changelog-repair"

# --- Provider ---------------------------------------------------------------
PROVIDER = "deepseek"                    # deepseek | anthropic | lm_studio
MODEL = "deepseek-v4-flash"              # deepseek-v4-flash | deepseek-v4-pro

# adaptive: thinking OFF until a recent failure, then ON. Matched on three
# proofs it accepted 43% of productive calls against 4% for disabled.
#
# `high_unless_stuck` also exists (effort `high` while converging) but was
# MEASURED AND REVERTED on ElGamal: same lemma, first 13 steps,
#     adaptive           median 121s   max  289s
#     high_unless_stuck  median 147s   max  914s
# It barely moves the typical step and triples the tail, with no evidence of
# better outcomes.
THINKING_MODE = "adaptive"               # disabled | enabled | adaptive |
                                         # high_unless_stuck
REASONING_EFFORT = None                  # deepseek: high|max
EMBED_MODEL = "text-embedding-nomic-embed-text-v1.5"

# --- Budget -----------------------------------------------------------------
MAX_TRIALS = 33
ADAPTIVE_MULTIPLIER = 2.5                # step budget = this x tactic lines
MIN_STEPS = 10
STUCK_LIMIT = 20
TOP_K_PREMISES = 10
LLM_MAX_TOKENS = 32768
# Per-request timeout, shared by the chat and embedding clients. 180 not 600:
# the OpenAI SDK retries twice, so 600 let one wedged call hold a trial for 30
# minutes. Transport errors are now retried rather than fatal, so failing fast
# is strictly better.
LLM_TIMEOUT_S = 180
COST_LIMIT_USD = 5.00

DATA_DIR = Path("data")
OUTPUT_DIR = None                        # None = auto-timestamped


## Build the config

**Run this before the run cell, always.** It mints a fresh `output_dir`; reusing a stale one overwrites a previous run in place.

In [ ]:
exp_config = ns.build_config(
    spec_name=SPEC_NAME, provider=PROVIDER, model=MODEL,
    thinking=THINKING_MODE, reasoning_effort=REASONING_EFFORT,
    embed_model=EMBED_MODEL, trials=MAX_TRIALS,
    adaptive_multiplier=ADAPTIVE_MULTIPLIER, min_steps=MIN_STEPS,
    stuck_limit=STUCK_LIMIT, top_k=TOP_K_PREMISES,
    llm_max_tokens=LLM_MAX_TOKENS, llm_timeout_s=LLM_TIMEOUT_S,
    cost_limit_usd=COST_LIMIT_USD, data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
)
agent = exp_config.agent
print(f"Spec            : {exp_config.spec_name}")
print(f"Provider/model  : {agent.llm_provider} / {agent.llm_model}")
print(f"Thinking/effort : {agent.llm_thinking} / {agent.llm_reasoning_effort}")
print(f"Output dir      : {exp_config.output_dir}")
print(f"Spend cap       : {agent.spend_budget.status() if agent.spend_budget else 'none (uncapped)'}")


## Preview: proof cases, shortest first

In [ ]:
from integration.experiment.corpora.joy import JoyCorpus

rows = ns.preview_cases(exp_config, DATA_DIR, JoyCorpus, MAX_TRIALS)
print(f"Available proofs: {len(rows)}   |   attempting: {min(MAX_TRIALS, len(rows))}\n")
print(f"{'#':<3} {'Name':<28} {'Lines':>6} {'Steps':>7}")
print("-" * 48)
for i, name, lines, steps in rows:
    print(f"{i:<3} {name:<28} {lines:>6} {steps:>7}{' <--' if i < MAX_TRIALS else ''}")


## Confirm paid usage

In [ ]:
from integration.agent.config import PAID_LLM_PROVIDERS
from integration.experiment.paid_confirm import (
    CONFIRMATION_PHRASE, format_paid_provider_warning,
)

CONFIRMED = agent.llm_provider not in PAID_LLM_PROVIDERS
if CONFIRMED:
    print(f"{agent.llm_provider}: not a paid provider, no confirmation needed.")
else:
    print(format_paid_provider_warning(config=agent, trials=MAX_TRIALS, informal=False))
    CONFIRMED = input().strip() == CONFIRMATION_PHRASE
    print("Confirmed." if CONFIRMED else "NOT confirmed - the run cell will refuse.")


## Run

In [ ]:
from integration.experiment.runner import run_experiment

exp_config.output_dir.mkdir(parents=True, exist_ok=True)
spec = ns.build_spec(exp_config, DATA_DIR)

if not CONFIRMED:
    raise RuntimeError(
        "Paid usage was not confirmed - run the confirmation cell above. "
        "No API call was made."
    )

result = run_experiment(spec, exp_config)

print(f"\nSpec      : {result.spec_name}   mode: {result.mode}")
print(f"Trials    : {result.trials_run} run, {result.trials_skipped} skipped")
print(f"Outcomes  : {result.successes} complete, {result.stuck} stuck, "
      f"{result.max_steps} max-steps, {result.errors} errors")
if result.estimated_cost:
    print(f"Cost      : ${result.estimated_cost['usd']:.6f}")
print(f"Output    : {result.output_dir}")


## Results

A bare "N complete" mostly reports that EasyCrypt still compiles the corpus, so replays and real model repairs are separated here.

In [ ]:
print("\n".join(ns.summarise(result)))

# net_tactics_vs_bootstrap: the only measure that caught the agent dismantling
# its own verified prefix. Empty for specs with no replayed prefix.
retained = ns.retained_tactics(result)
if retained:
    print("\nTactics retained vs the bootstrap prefix:")
    print("\n".join(retained))


## Per-failure diagnostics

In [ ]:
import json
from collections import Counter
from integration.agent.ec_errors import classify_error, strip_warning_lines

kinds, accepted, noops = Counter(), 0, 0
for trial_dir in sorted((result.output_dir / "trials").iterdir()):
    log = trial_dir / "agent_log.json"
    if not log.is_file():
        continue
    for e in json.loads(log.read_text()).get("events", []):
        if e.get("event") != "iteration" or e.get("action") != "tactic":
            continue
        if e.get("outcome") in ("accepted", "complete"):
            accepted += 1
        elif e.get("outcome") == "no_op":
            noops += 1
        elif e.get("outcome") == "failed" and e.get("error"):
            # strip_warning_lines FIRST: EasyCrypt prints file-level warnings
            # before the [critical] line, and taking line 0 raw attributes the
            # failure to a warning.
            kinds[classify_error(strip_warning_lines(e["error"])).kind] += 1

print(f"Tactics accepted by the model : {accepted}")
print(f"Tactics that changed NOTHING  : {noops}"
      f"  ({100*noops/max(1, accepted+noops):.0f}% of productive steps)")
print("\nFailure kinds:")
for kind, n in kinds.most_common():
    print(f"  {kind:<24} {n}")
